<a href="https://colab.research.google.com/github/Leo278V/Final-Assignment-PDS/blob/Natural-Language-Processing-Model/Advanced_NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:

data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Natural-Language-Processing-Model/df_profiles_cleansed.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()


,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658


In [3]:
#Check for missing values

df_profiles.isna().sum()


,0
organization,0
position,0
startDate,0
endDate,0
status,0
department,0
seniority,0
person_id,0
job_count,0
job_duration_years,0


In [4]:
#Check for type of values

df_profiles["status"].value_counts(dropna=False)


,count
status,
INACTIVE,1897
ACTIVE,718


In [5]:
# Seniority CSV
data_path_seniority = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/seniority-v2.csv"
df_seniority = pd.read_csv(data_path_seniority)
df_seniority.head()

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior


In [6]:
#Checking for missing values
df_seniority.isna().sum()


,0
text,0
label,0


In [7]:
df_seniority.shape

(9428, 2)

In [8]:
# Department CSV
data_path_department = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/department-v2.csv"
df_department = pd.read_csv(data_path_department)
df_department.head()

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management


In [9]:
#Checking for missing values
df_department.isna().sum()

,0
text,0
label,0


In [10]:
df_department.shape

(10145, 2)

In [11]:
# Building Match Text

df_profiles["match_text"] = (
    df_profiles["position"].fillna("").astype(str).str.strip()
    + " at "
    + df_profiles["organization"].fillna("").astype(str).str.strip()
).str.strip()


In [12]:
# Loading Embedding Model

!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# Setting up Vectors

def build_label_centroids(df_labels, text_col="text", label_col="label", batch_size=512):
    df_tmp = df_labels.dropna(subset=[text_col, label_col]).copy()
    df_tmp[text_col] = df_tmp[text_col].astype(str).str.strip()
    df_tmp[label_col] = df_tmp[label_col].astype(str).str.strip()

    texts = df_tmp[text_col].tolist()
    labels = df_tmp[label_col].tolist()

    emb = model.encode(texts, normalize_embeddings=True, batch_size=batch_size, show_progress_bar=True)

    # pro Label Mittelwert bilden
    label_to_vecs = {}
    for vec, lab in zip(emb, labels):
        label_to_vecs.setdefault(lab, []).append(vec)

    label_names = sorted(label_to_vecs.keys())
    centroids = np.vstack([np.mean(label_to_vecs[lab], axis=0) for lab in label_names])

    # centroids normalisieren (wichtig für cosine similarity)
    centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

    return label_names, centroids


In [14]:
# Predicting only active jobs

df_active = df_profiles[df_profiles["status"] == "ACTIVE"].copy()
texts = df_active["match_text"].fillna("").astype(str).tolist()


In [15]:
# Prediction Function

def predict_from_centroids(texts, label_names, centroids, batch_size=256):
    preds, scores = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, normalize_embeddings=True, batch_size=batch_size, show_progress_bar=False)
        sims = cosine_similarity(emb, centroids)   # (batch, n_labels)
        idx = np.argmax(sims, axis=1)
        best = sims[np.arange(len(batch)), idx]
        preds.extend([label_names[j] for j in idx])
        scores.extend(best.tolist())
    return preds, scores


In [17]:
# Prediciting Department and Seniority
dep_label_names, dep_centroids = build_label_centroids(df_department, text_col="text", label_col="label")
sen_label_names, sen_centroids = build_label_centroids(df_seniority, text_col="text", label_col="label")

df_active["pred_department"], df_active["pred_department_score"] = predict_from_centroids(
    texts, dep_label_names, dep_centroids
)

df_active["pred_seniority"], df_active["pred_seniority_score"] = predict_from_centroids(
    texts, sen_label_names, sen_centroids
)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [19]:
# EValuation

dep_eval = df_active.dropna(subset=["department", "pred_department"])
print("=== Department (ACTIVE) ===")
print(classification_report(dep_eval["department"], dep_eval["pred_department"]))

sen_eval = df_active.dropna(subset=["seniority", "pred_seniority"])
print("=== Seniority (ACTIVE) ===")
print(classification_report(sen_eval["seniority"], sen_eval["pred_seniority"]))

=== Department (ACTIVE) ===
                        precision    recall  f1-score   support

        Administrative       0.05      0.30      0.09        23
  Business Development       0.14      0.40      0.20        20
            Consulting       0.26      0.53      0.35        45
      Customer Support       0.18      0.29      0.22         7
       Human Resources       0.27      0.47      0.35        19
Information Technology       0.39      0.32      0.35        69
             Marketing       0.33      0.58      0.42        24
                 Other       0.68      0.14      0.23       405
    Project Management       0.46      0.46      0.46        39
            Purchasing       0.02      0.06      0.03        16
                 Sales       0.21      0.49      0.29        51

              accuracy                           0.26       718
             macro avg       0.27      0.37      0.27       718
          weighted avg       0.50      0.26      0.27       718

=== Senio

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
